# Regressió ML amb Sklearn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Càrrega de dades

Aquestes dades han d'estar netes i preparades per aplicar ML.

En aquest exemple de `tips`, volem predir la propina (`tip`) que pagarà un client.

In [ ]:
df = sns.load_dataset('tips')
df.head()

## Pipeline de Feature Engineering

Defineix els passos de preprocessament per als models que ho necessiten.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

In [ ]:
onehot_pipeline = Pipeline([
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore')),
])

In [ ]:
onehot_features = ['sex', 'smoker', 'day', 'time']

preprocessor = ColumnTransformer(
    transformers=[
        ('onehot', onehot_pipeline, onehot_features),
    ],
    remainder='passthrough',
)

## Entrenament / Test

- Podem emular dades no vistes dividint les nostres dades en conjunts d'**entrenament** i de **test**.
    - Usem el conjunt d'entrenament per ajustar (entrenar) el nostre model.
    - Usem el conjunt de test per avaluar el rendiment del model amb dades no vistes.

- Les mides de test habituals són del 10-30% de les dades inicials.
    - Els conjunts de dades grans poden requerir un percentatge menor (p. ex., 10-20%).
    - Els conjunts de dades petits poden necessitar un conjunt de test més gran per garantir que sigui representatiu (p. ex., 30-40%).

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
# Crear la variable X amb les característiques predictores (sense l'objectiu!)
X = df.drop('tip', axis=1)

# Separar l'objectiu en la variable `y`
y = df['tip']

# Divisió Entrenament / Test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,  # 20% de les dades en el test
    shuffle=True,
    random_state=42
)

## Entrenament del model

In [ ]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

### Baseline

- Entrena i avalua el rendiment d'un model de referència (baseline).
- Proporciona una mesura inicial del rendiment.
- Qualsevol model que desenvolupis hauria de superar aquesta línia base simple.

In [ ]:
from sklearn.dummy import DummyRegressor

In [ ]:
# El baseline predirà sempre el valor mitjà de la columna target
bl = DummyRegressor(strategy='mean')

# Entrenar
bl.fit(X_train, y_train)

# Predir el conjunt d'entrenament
y_pred_train = bl.predict(X_train)

# Calcular els errors
mae = mean_absolute_error(y_train, y_pred_train)
rmse = root_mean_squared_error(y_train, y_pred_train)

In [ ]:
print(f'Train MAE: {mae:.1f}')
print(f'Train RMSE: {rmse:.1f}')

### Regressió lineal

- Cada vegada que entrenem la regressió lineal,
- el model aprèn els coeficients $w_i$ (en funció de les característiques $x_i$):

    $\hat{y} = w_0 + w_1 \cdot x_1 + w_2 \cdot x_2 + \text{...} + w_n \cdot x_n$


- de manera que les prediccions $\hat{y}$ minimitzin $MSE(y, \hat{y})$.

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
# Definim una Pipeline que primer processarà les dades i llavors aplicarà LR
lr = Pipeline([
    ('preprocessor', preprocessor),
    ('lr', LinearRegression(n_jobs=-1))
])

# Entrenar
lr.fit(X_train, y_train)

# Predir el conjunt d'entrenament
y_pred_train = lr.predict(X_train)

# Calcular els errors
mae = mean_absolute_error(y_train, y_pred_train)
rmse = root_mean_squared_error(y_train, y_pred_train)

In [ ]:
print(f'Train MAE: {mae:.1f}')
print(f'Train RMSE: {rmse:.1f}')

## Test

Els models ja han estat entrenats amb `X_train`. Vegem com funcionen amb dades que mai han vist (`X_test`).

In [ ]:
'''# Predir les dades de test (no vistes)
y_test_bl = bl.predict(X_test)
y_test_lr = lr.predict(X_test)

# Calcular el MAE de test
mae_test_bl = mean_absolute_error(y_test, y_test_bl)
mae_test_lr = mean_absolute_error(y_test, y_test_lr)

# Calcular el RMSE de test
rmse_test_bl = root_mean_squared_error(y_test, y_test_bl)
rmse_test_lr = root_mean_squared_error(y_test, y_test_lr)'''

In [ ]:
# Visualitzar els resultats

results_df = pd.DataFrame(
    {
        'MAE': [mae_test_bl, mae_test_lr],
        'RMSE': [rmse_test_bl, rmse_test_lr]
    },
    index=['Baseline', 'LR'],
)

results_df.plot(
    kind='bar',
    title='Rendiment en el test',
    ylabel='Error'
);